In [6]:
from __future__ import annotations
from dataclasses import dataclass
from math import ceil
from typing import Dict, List, Optional


@dataclass(frozen=True)
class RainScenario:
    key: str
    name: str
    treatment_rain_mmph: float
    runoff_coeff: float

SCENARIOS: Dict[str, RainScenario] = {
    "AUCKLAND": RainScenario("AUCKLAND", "Auckland", 10.0, 0.95),
    "CHRISTCHURCH": RainScenario("CHRISTCHURCH", "Christchurch", 5.0, 1.00),
    "REST_OF_NZ": RainScenario("REST_OF_NZ", "Rest of NZ", 10.0, 1.00),
}


def normalize_region(region: str) -> str:
    r = region.strip().upper().replace("-", "_").replace(" ", "_")
    if r in ("AKL", "AUCKLAND"):
        return "AUCKLAND"
    if r in ("CHC", "CHRISTCHURCH"):
        return "CHRISTCHURCH"
    if r in ("REST_OF_NZ", "REST", "OTHER", "ELSEWHERE", "NZ", "NEW_ZEALAND"):
        return "REST_OF_NZ"
    raise ValueError("Region must be: Auckland / Christchurch / Rest of NZ")


@dataclass(frozen=True)
class ProductOption:
    code: str
    name: str
    family: str
    capacity_lps_per_unit: float
    unit_cost_index: float


PRODUCTS: List[ProductOption] = [

    ProductOption("ATLAN_FULL", "Atlan Filter (Full Size)", "ATLAN", 3.0, 1.00),
    ProductOption("ATLAN_HALF", "Atlan Filter (Half Size)", "ATLAN", 1.5, 0.70),

    ProductOption("FLOW_400", "Flow Filter (400 Series)", "FLOW", 7.5, 1.60),
    ProductOption("FLOW_1500", "Flow Filter (1500 Series)", "FLOW", 15.0, 3.10),

    ProductOption("FLOWGUARD", "FlowGuard", "FLOWGUARD", 10.0, 2.40),
]


def eligible_products(region_key: str) -> List[ProductOption]:

    if region_key == "AUCKLAND":
        return [p for p in PRODUCTS if p.family in ("ATLAN", "FLOWGUARD")]
    return list(PRODUCTS)


def treatment_flow_lps(area_m2: float, scenario: RainScenario) -> float:

    if area_m2 <= 0:
        raise ValueError("Impervious area must be > 0 m²")
    return (area_m2 * scenario.treatment_rain_mmph * scenario.runoff_coeff) / 3600.0


@dataclass(frozen=True)
class Selection:
    product: ProductOption
    units: int
    total_capacity_lps: float
    total_cost_index: float
    notes: List[str]


def choose_cheapest(required_lps: float, region_key: str) -> Selection:
    candidates: List[Selection] = []

    for p in eligible_products(region_key):
        units = max(1, ceil(required_lps / p.capacity_lps_per_unit))
        total_cap = units * p.capacity_lps_per_unit
        total_cost = units * p.unit_cost_index

        notes: List[str] = []

        if p.code == "FLOW_1500" and required_lps <= 7.5:
            notes.append("Guardrail: 1500 Series usually unnecessary below 7.5 L/s (prefer 400 Series).")
        if p.code == "ATLAN_HALF":
            notes.append("Guardrail: Half-size should be used only where form-factor constraints apply.")

        candidates.append(Selection(p, units, total_cap, total_cost, notes))

    candidates.sort(key=lambda x: (x.total_cost_index, x.units, -x.product.capacity_lps_per_unit))
    return candidates[0]


def force_product(required_lps: float, region_key: str, product_code: str) -> Selection:
    allowed = {p.code: p for p in eligible_products(region_key)}
    code = product_code.strip().upper()
    if code not in allowed:
        raise ValueError(f"Product {code} not eligible in {SCENARIOS[region_key].name}. "
                         f"Allowed: {sorted(allowed.keys())}")

    p = allowed[code]
    units = max(1, ceil(required_lps / p.capacity_lps_per_unit))
    return Selection(
        product=p,
        units=units,
        total_capacity_lps=units * p.capacity_lps_per_unit,
        total_cost_index=units * p.unit_cost_index,
        notes=[]
    )


def print_report(
    *,
    project: str,
    region_key: str,
    area_m2: float,
    selection: Selection,
    scenario: RainScenario,
    required_lps: float
) -> None:
    print("\n" + "=" * 68)
    print("STORMWATER TREATMENT SIZING (PYTHON DEMO)".center(68))
    print("=" * 68)
    print(f"Project: {project}")
    print(f"Region:  {scenario.name}")
    print("-" * 68)
    print("Inputs")
    print(f"  Impervious area (m²):     {area_m2:,.0f}")
    print(f"  Treatment rainfall (mm/hr): {scenario.treatment_rain_mmph:g}")
    print(f"  Runoff coefficient:       {scenario.runoff_coeff:g}")
    print("-" * 68)
    print("Calculated")
    print(f"  Treatment flow (L/s):     {required_lps:.2f}")
    print("  Bypass:                  Any flow above treatment flow may bypass (site-specific).")
    print("-" * 68)
    print("Selected Treatment System")
    print(f"  Product:                 {selection.product.name} ({selection.product.code})")
    print(f"  Capacity per unit (L/s): {selection.product.capacity_lps_per_unit:g}")
    print(f"  Units required:          {selection.units}  (rounded up)")
    print(f"  Total capacity (L/s):    {selection.total_capacity_lps:.2f}")
    print(f"  Cost index (demo):       {selection.total_cost_index:.2f}")
    print("-" * 68)

    eligible = eligible_products(region_key)
    print("Guardrails (by region)")
    print(f"  Eligible product families: {sorted({p.family for p in eligible})}")
    print(f"  Eligible product codes:    {sorted({p.code for p in eligible})}")

    if selection.notes:
        print("\nNotes")
        for n in selection.notes:
            print(f"  - {n}")

    if region_key == "AUCKLAND":
        print("\nDesign note (Auckland)")
        print("  - 10 mm/hr is used as the treatment rainfall in this model (per internal spreadsheet logic).")

    print("=" * 68 + "\n")

def run_demo() -> None:
    print("\nStormwater Calculator Demo (Python)")
    print("Type ENTER to accept defaults.\n")

    project = input("Project name [Demo Site]: ").strip() or "Demo Site"
    region_in = input("Region (Auckland / Christchurch / Rest of NZ) [Auckland]: ").strip() or "Auckland"
    region_key = normalize_region(region_in)

    area_str = input("Impervious area m² [1500]: ").strip() or "1500"
    area_m2 = float(area_str)

    scenario = SCENARIOS[region_key]
    required_lps = treatment_flow_lps(area_m2, scenario)

    print("\nOptional: force a product code (else auto-cheapest eligible).")
    print("Examples: ATLAN_FULL, ATLAN_HALF, FLOW_400, FLOW_1500, FLOWGUARD")
    force_code = input("Force product code [none]: ").strip()

    if force_code:
        selection = force_product(required_lps, region_key, force_code)
    else:
        selection = choose_cheapest(required_lps, region_key)

    print_report(
        project=project,
        region_key=region_key,
        area_m2=area_m2,
        selection=selection,
        scenario=scenario,
        required_lps=required_lps
    )


if __name__ == "__main__":
    run_demo()


Stormwater Calculator Demo (Python)
Type ENTER to accept defaults.

Project name [Demo Site]: Project C
Region (Auckland / Christchurch / Rest of NZ) [Auckland]: Christchurch
Impervious area m² [1500]: 3000

Optional: force a product code (else auto-cheapest eligible).
Examples: ATLAN_FULL, ATLAN_HALF, FLOW_400, FLOW_1500, FLOWGUARD
Force product code [none]: 

             STORMWATER TREATMENT SIZING (PYTHON DEMO)              
Project: Project C
Region:  Christchurch
--------------------------------------------------------------------
Inputs
  Impervious area (m²):     3,000
  Treatment rainfall (mm/hr): 5
  Runoff coefficient:       1
--------------------------------------------------------------------
Calculated
  Treatment flow (L/s):     4.17
  Bypass:                  Any flow above treatment flow may bypass (site-specific).
--------------------------------------------------------------------
Selected Treatment System
  Product:                 Flow Filter (400 Series) (FLOW_40